# Autonomous Crypto Trading Lab

Run the single cell below. Add your Kimi API key to Colab Secrets as `MOONSHOT_API_KEY` first. The key is never written to GitHub.


In [ ]:
import pathlib, subprocess, sys, os
from getpass import getpass

REPO_URL = 'https://github.com/betaanoiar1-gif/autonomous-crypto-trading-lab.git'
REPO_DIR = '/content/autonomous_crypto_trading_lab'

print('=== Autonomous Crypto Trading Lab ===')

try:
    from google.colab import userdata
    api_key = userdata.get('MOONSHOT_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = getpass('Enter Kimi API key (not saved to GitHub): ')

if not api_key:
    raise RuntimeError('A Kimi API key is required.')

os.environ['MOONSHOT_API_KEY'] = api_key

if pathlib.Path(REPO_DIR).exists():
    print('Updating project...')
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
else:
    print('Cloning project...')
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)

print('Installing dependencies...')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{REPO_DIR}/requirements.txt'], check=True)
sys.path.insert(0, REPO_DIR)

print('Checking Kimi connection...')
from lab.kimi_agent import KimiAgent
health = KimiAgent().healthcheck()
print(f"Kimi: {'OK' if health['ok'] else 'FAILED'} | model={health['model']}")

if not health['ok']:
    raise RuntimeError('Kimi healthcheck failed.')

print('Launching lab...')
subprocess.run([sys.executable,'-m','lab.orchestrator'], cwd=REPO_DIR, check=True)
print('=== Launch complete ===')
